In [ ]:
# Cell 1: Install required packages
!pip install PyGithub GitPython requests

In [ ]:
# Cell 2: Import libraries
import os
import shutil
import requests
import json
from git import Repo
from github import Github
from getpass import getpass
import time
from IPython.display import clear_output
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# Cell 3: Configuration - Enter your GitHub credentials
print("=" * 60)
print("GITHUB REPOSITORY TRANSFER TOOL")
print("=" * 60)

# Get credentials securely
old_username = "************"
old_token = "***********"

new_username = "***********"
new_token = "***************"

# Store in variables
print("\n✅ Credentials saved securely!")

GITHUB REPOSITORY TRANSFER TOOL

✅ Credentials saved securely!


In [ ]:
# Cell 4: Test connections to both accounts
def test_connection(username, token, account_type="Old"):
    """Test GitHub API connection"""
    headers = {"Authorization": f"token {token}"}
    response = requests.get("https://api.github.com/user", headers=headers)

    if response.status_code == 200:
        user_data = response.json()
        print(f"✅ {account_type} account connected: {user_data['login']}")
        print(f"   • Name: {user_data.get('name', 'N/A')}")
        print(f"   • Public repos: {user_data['public_repos']}")
        print(f"   • Private repos: {user_data.get('total_private_repos', 0)}")
        return True
    else:
        print(f"❌ Failed to connect to {account_type} account")
        print(f"   Error: {response.status_code} - {response.json().get('message', 'Unknown error')}")
        return False

print("Testing connections...\n")
old_ok = test_connection(old_username, old_token, "Old")
new_ok = test_connection(new_username, new_token, "New")

if not (old_ok and new_ok):
    print("\n❌ Connection failed! Please check your credentials and try again.")

In [ ]:
# Cell 5: Fetch all repositories from old account
def get_all_repos(token):
    """Fetch all repositories including private ones"""
    headers = {"Authorization": f"token {token}"}
    all_repos = []
    page = 1

    print("Fetching repositories...")

    while True:
        url = f"https://api.github.com/user/repos?per_page=100&page={page}&type=all"
        response = requests.get(url, headers=headers)

        if response.status_code != 200 or not response.json():
            break

        repos = response.json()
        all_repos.extend(repos)

        # Progress indicator
        print(f"  Found {len(all_repos)} repos so far...", end='\r')

        if len(repos) < 100:  # Last page
            break

        page += 1
        time.sleep(0.5)  # Rate limit protection

    print(f"\n✅ Total repositories found: {len(all_repos)}")

    # Display summary
    public_repos = [r for r in all_repos if not r['private']]
    private_repos = [r for r in all_repos if r['private']]

    print(f"   • Public repos: {len(public_repos)}")
    print(f"   • Private repos: {len(private_repos)}")

    return all_repos

if old_ok:
    repos = get_all_repos(old_token)

    # Show repos list
    print("\n" + "=" * 60)
    print("REPOSITORIES TO TRANSFER:")
    print("=" * 60)
    for i, repo in enumerate(repos, 1):
        visibility = "🔒 Private" if repo['private'] else "🌐 Public"
        size_kb = repo['size']
        size_mb = size_kb / 1024 if size_kb > 0 else 0
        print(f"{i:3d}. {repo['name']:<40s} {visibility} ({size_mb:.1f} MB)")

In [ ]:
# Cell 6: Create repo on new account
def create_repo_on_new_account(repo_name, is_private, description=""):
    """Create repository on new account"""
    headers = {
        "Authorization": f"token {new_token}",
        "Accept": "application/vnd.github.v3+json"
    }

    url = "https://api.github.com/user/repos"
    data = {
        "name": repo_name,
        "private": is_private,
        "description": description,
        "auto_init": False  # Don't initialize with README
    }

    # Check if repo already exists
    check_url = f"https://api.github.com/repos/{new_username}/{repo_name}"
    check_response = requests.get(check_url, headers=headers)

    if check_response.status_code == 200:
        return "exists"

    response = requests.post(url, headers=headers, json=data)

    if response.status_code == 201:
        return "created"
    else:
        return f"error: {response.json().get('message', 'Unknown error')}"

# Test with a sample
if old_ok and new_ok:
    test_repo = repos[0]
    result = create_repo_on_new_account(
        test_repo['name'],
        test_repo['private'],
        test_repo.get('description', '')
    )
    print(f"Test creation of '{test_repo['name']}': {result}")

Test creation of 'abhlshekgupta': created


In [ ]:
# Cell 8: Transfer all repositories (THE MAIN EXECUTION CELL)
def transfer_all_repos(repos, start_from=0):
    """Transfer all repositories with progress tracking"""
    total = len(repos)
    successful = 0
    failed = 0
    skipped = 0

    print("=" * 60)
    print(f"STARTING TRANSFER OF {total} REPOSITORIES")
    print("=" * 60)

    start_time = time.time()

    for i in range(start_from, total):
        repo = repos[i]

        # Check if repo should be skipped (e.g., already exists)
        result = transfer_single_repo(repo, i + 1, total)

        if result:
            successful += 1
        else:
            failed += 1

        # Progress update
        elapsed = time.time() - start_time
        avg_time = elapsed / (i - start_from + 1)
        remaining = avg_time * (total - i - 1)

        print(f"\n📊 Progress: {i+1}/{total} | "
              f"✅ {successful} | ❌ {failed} | "
              f"⏱️ Est. remaining: {remaining:.1f}s")
        print("=" * 60)

        # Rate limiting protection
        time.sleep(2)

    # Final summary
    print("\n\n" + "=" * 60)
    print("TRANSFER COMPLETE!")
    print("=" * 60)
    print(f"✅ Successfully transferred: {successful}")
    print(f"❌ Failed transfers: {failed}")
    print(f"⏱️ Total time: {time.time() - start_time:.1f} seconds")
    print(f"\n🎉 Check your new account: https://github.com/{new_username}")

    return successful, failed

# Safety check before running
print("⚠️  READY TO TRANSFER ALL REPOSITORIES")
print(f"   From: {old_username}")
print(f"   To: {new_username}")
print(f"   Total repos: {len(repos)}")
print("\nMake sure you have enough storage space in Colab!")
print("Temporary files will be stored in /tmp/ and cleaned up.")

confirm = input("\nType 'YES' to proceed with transfer: ")
if confirm == "YES":
    successful, failed = transfer_all_repos(repos)
else:
    print("Transfer cancelled.")